# Helpdesk — Old vs Improved

Single-source comparison: runs MC suffix sampling for both checkpoints (skipping any side that
already has chunked outputs), evaluates the same metric set Henryk uses in
`evaluation_metric_notebooks/improved/`, then plots them side by side.

**Pair:** `henryk/helpdesk` &nbsp;·&nbsp; **Activity key:** `Activity` &nbsp;·&nbsp; **Samples/case:** `100`

## How to run

1. Confirm the paths in the parameter cell point at the right checkpoints + test pickles.
2. Execute top-to-bottom. Sampling is skipped automatically if `SAMPLED_DIR_*` already contains `results_part_*.pkl`.
3. The bottom cells (table + overlay plot) tolerate either side being `None` — useful while the improved
   checkpoint is still being trained.


## Parameters

In [1]:
# === PARAMETER CELL ===
# Tweak paths and knobs here. Everything below should run unchanged.
from pathlib import Path

PAIR_DIR = Path('.').resolve()   # the dir containing this notebook

# --- model checkpoints (None = side not yet trained / placeholder) ---
MODEL_OLD_PATH      = PAIR_DIR / 'old/Training/Helpdesk_full_grad_norm_philipp_4layer_philipp_final_run.pkl'
MODEL_IMPROVED_PATH = PAIR_DIR / 'improved/Training/pkl/Helpdesk_full_grad_norm_improved_henryk.pkl'

# --- encoded test pickles ---
TEST_PKL_OLD      = PAIR_DIR / 'old/Loader/helpdesk_all_5_test.pkl'
TEST_PKL_IMPROVED = PAIR_DIR / 'improved/Loader/pkl/helpdesk_all_5_test.pkl'

# --- where the chunked sampling results land (also where batch_evaluate reads from) ---
SAMPLED_DIR_OLD      = PAIR_DIR / 'evaluation_results/old'
SAMPLED_DIR_IMPROVED = PAIR_DIR / 'evaluation_results/improved'

# --- comparison output ---
COMPARISON_PKL = PAIR_DIR / 'helpdesk_old_vs_improved.pkl'
CAPTION        = 'Helpdesk'

# --- sampling knobs (ProbabilisticEvaluation kwargs) ---
CONCEPT_NAME        = 'Activity'
ALL_CAT             = ['Activity', 'Resource']
ALL_NUM             = ['case_elapsed_time', 'event_elapsed_time']
GROWING_NUM_VALUES  = ['case_elapsed_time']
NUM_PROCESSES       = 16
SAMPLES_PER_CASE    = 100
SAVE_EVERY          = 50
RANDOM_ORDER        = True
USE_VARIANCE_CAT    = True
USE_VARIANCE_NUM    = True
SAMPLE_ARGMAX       = False

# --- metric knobs ---
ACTIVITY_KEY      = 'Activity'
EVENT_LABEL_LIST  = ['Assign seriousness', 'Take in charge ticket', 'Resolve ticket', 'Closed', 'Insert ticket', 'Wait', 'Create SW anomaly', 'Require upgrade', 'VERIFIED', 'DUPLICATE', 'Resolve SW anomaly', 'Schedule intervention', 'RESOLVED', 'INVALID']
VALUE_FACTOR_TIME = 86400   # 3600*24 reports remaining-time in days


## Setup

In [2]:
import sys, importlib
from pathlib import Path

# Reach `src/` so `model.*`, `src.evaluation_metrics.*` resolve.
_REPO_ROOT = Path('.').resolve()
while not (_REPO_ROOT / 'src').is_dir() and _REPO_ROOT != _REPO_ROOT.parent:
    _REPO_ROOT = _REPO_ROOT.parent
for p in (str(_REPO_ROOT), str(_REPO_ROOT / 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

# Reach the _shared helper module.
_SHARED = _REPO_ROOT / 'src' / 'interpretability' / 'improved_pipeline' / '_shared'
if str(_SHARED) not in sys.path:
    sys.path.insert(0, str(_SHARED))

import comparison_helpers
importlib.reload(comparison_helpers)
from comparison_helpers import (
    SamplingConfig, ensure_sampled, default_metric_set, evaluate_dir,
    comparison_table, plot_overlay, save_results,
)


## MC suffix sampling (skipped per side if chunks already exist)

In [3]:
sampling_cfg = SamplingConfig(
    concept_name=CONCEPT_NAME,
    growing_num_values=GROWING_NUM_VALUES,
    all_cat=ALL_CAT,
    all_num=ALL_NUM,
    num_processes=NUM_PROCESSES,
    samples_per_case=SAMPLES_PER_CASE,
    sample_argmax=SAMPLE_ARGMAX,
    use_variance_cat=USE_VARIANCE_CAT,
    use_variance_num=USE_VARIANCE_NUM,
    random_order=RANDOM_ORDER,
    save_every=SAVE_EVERY,
)

if MODEL_OLD_PATH and TEST_PKL_OLD:
    ensure_sampled(MODEL_OLD_PATH, TEST_PKL_OLD, SAMPLED_DIR_OLD, sampling_cfg)
else:
    print('[skip] OLD side: missing MODEL_OLD_PATH or TEST_PKL_OLD')

if MODEL_IMPROVED_PATH and TEST_PKL_IMPROVED:
    ensure_sampled(MODEL_IMPROVED_PATH, TEST_PKL_IMPROVED, SAMPLED_DIR_IMPROVED, sampling_cfg)
else:
    print('[skip] IMPROVED side: missing MODEL_IMPROVED_PATH or TEST_PKL_IMPROVED')


[sample] loading model from /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/helpdesk/old/Training/Helpdesk_full_grad_norm_philipp_4layer_philipp_final_run.pkl
Data set categories:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15}), ('Resource', 24, {'EOS': 1, 'Value 1': 2, 'Value 10': 3, 'Value 11': 4, 'Value 12': 5, 'Value 13': 6, 'Value 14': 7, 'Value 15': 8, 'Value 16': 9, 'Value 17': 10, 'Value 18': 11, 'Value 19': 12, 'Value 2': 13, 'Value 20': 14, 'Value 21': 15, 'Value 22': 16, 'Value 3': 17, 'Value 4': 18, 'Value 5': 19, 'Value 6': 20, 'Value 7': 21, 'Value 8': 22, 'Value 9': 23}), ('Variant index', 175, {'1.0': 1, '10.0': 2, '100.0': 3

  0%|          | 0/916 [00:00<?, ?it/s]

[sample] wrote 50 prefixes -> results_part_050.pkl
[sample] wrote 50 prefixes -> results_part_100.pkl
[sample] wrote 50 prefixes -> results_part_150.pkl
[sample] wrote 50 prefixes -> results_part_200.pkl
[sample] wrote 50 prefixes -> results_part_250.pkl
[sample] wrote 50 prefixes -> results_part_300.pkl
[sample] wrote 50 prefixes -> results_part_350.pkl
[sample] wrote 50 prefixes -> results_part_400.pkl
[sample] wrote 50 prefixes -> results_part_450.pkl
[sample] wrote 50 prefixes -> results_part_500.pkl
[sample] wrote 50 prefixes -> results_part_550.pkl
[sample] wrote 50 prefixes -> results_part_600.pkl
[sample] wrote 50 prefixes -> results_part_650.pkl
[sample] wrote 50 prefixes -> results_part_700.pkl
[sample] wrote 50 prefixes -> results_part_750.pkl
[sample] wrote 50 prefixes -> results_part_800.pkl
[sample] wrote 50 prefixes -> results_part_850.pkl
[sample] wrote 50 prefixes -> results_part_900.pkl
[sample] wrote 50 prefixes -> results_part_950.pkl
[sample] wrote 50 prefixes -> r

  0%|          | 0/916 [00:00<?, ?it/s]

KeyError: 13

## Build metric set

In [ ]:
metrics = default_metric_set(
    activity_key=ACTIVITY_KEY,
    event_label_list=EVENT_LABEL_LIST,
    value_factor_time=VALUE_FACTOR_TIME,
)
print(f'metric set has {len(metrics)} entries')


## Evaluate sampled outputs

In [ ]:
res_old, counts_old = (None, None)
res_improved, counts_improved = (None, None)

if SAMPLED_DIR_OLD.is_dir() and any(SAMPLED_DIR_OLD.glob('results_part_*.pkl')):
    res_old, counts_old = evaluate_dir(SAMPLED_DIR_OLD, metrics)
else:
    print('[skip] OLD eval: no chunks under', SAMPLED_DIR_OLD)

if SAMPLED_DIR_IMPROVED.is_dir() and any(SAMPLED_DIR_IMPROVED.glob('results_part_*.pkl')):
    res_improved, counts_improved = evaluate_dir(SAMPLED_DIR_IMPROVED, metrics)
else:
    print('[skip] IMPROVED eval: no chunks under', SAMPLED_DIR_IMPROVED)


## Side-by-side metric table

In [ ]:
import pandas as pd
df = comparison_table(res_old, res_improved)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
df


## Overlay plots

In [ ]:
plot_overlay(res_old, res_improved, counts_old, counts_improved, caption=CAPTION, pgf=False)


## Save comparison pickle

In [ ]:
save_results(
    COMPARISON_PKL,
    res_old=res_old, counts_old=counts_old,
    res_improved=res_improved, counts_improved=counts_improved,
    config_old=sampling_cfg, config_improved=sampling_cfg,
)


## Notes

- The comparison pickle written in the last cell holds both `(res_raw, counts)` pairs and the sampling configs that produced them. Re-load it later with `comparison_helpers.load_results(path)`.
- To force re-sampling, pass `force=True` into the explicit `ensure_sampled` calls (or just delete the chunked output dir).
- For dataset-specific metric tweaks, copy `default_metric_set` into a cell and edit it; the rest of the pipeline only cares that `metrics` is a `dict[str, metric]`.
